# Metric view join cardinality test

Runs after `register_each_mv`. For every metric view with `joins`, compares `COUNT(*)` of the fact grain (`source`) with `COUNT(*)` after adding each join in turn (parents before nested joins, `LEFT JOIN` like a metric view). A larger count means a join fans out (it is not many-to-one), so measures over the fact would be inflated: the task fails and names the first inflating join.

The check reads the source tables, not the registered views, so it also catches joins whose `rely.at_most_one_match` claim is wrong.

**Widgets:** `bundle_root`, `metric_views_dir`, `source_catalog`, `source_schema`, `target_catalog`, `target_schema` (all required; substituted into the YAML like registration does).

In [ ]:
%pip install --upgrade databricks-sdk pyyaml
dbutils.library.restartPython()

In [ ]:
WIDGETS = ["bundle_root", "metric_views_dir", "source_catalog", "source_schema", "target_catalog", "target_schema"]
for w in WIDGETS:
    dbutils.widgets.text(w, "")
args = {w: dbutils.widgets.get(w).strip() for w in WIDGETS}

missing = [k for k, v in args.items() if not v]
if missing:
    raise ValueError(f"Required widget(s) not set: {missing}")
for k, v in args.items():
    print(f"{k}={v}")

In [ ]:
import importlib
import json
import re
import sys
import types
from pathlib import Path


def load_sdk(root: str):
    """Import mv_yaml and manifest from <root>/src/genie_ontology_modeled_knowledge_sdk_vendored/.

    The vendored files are verbatim copies of src/genie_ontology_modeled_knowledge_sdk/ (they import
    each other as genie_ontology_modeled_knowledge_sdk.*), so the vendored folder is registered as
    the genie_ontology_modeled_knowledge_sdk package. Falls back to an installed genie_ontology_modeled_knowledge_sdk.
    """
    vendored = Path(root) / "src" / "genie_ontology_modeled_knowledge_sdk_vendored"
    if (vendored / "mv_yaml.py").is_file():
        pkg = types.ModuleType("genie_ontology_modeled_knowledge_sdk")
        pkg.__path__ = [str(vendored)]
        sys.modules["genie_ontology_modeled_knowledge_sdk"] = pkg
    try:
        return (importlib.import_module("genie_ontology_modeled_knowledge_sdk.mv_yaml"),
                importlib.import_module("genie_ontology_modeled_knowledge_sdk.manifest"))
    except ImportError as exc:
        raise ImportError(
            f"genie_ontology_modeled_knowledge_sdk not found: expected the vendored library in {vendored} "
            "(scaffold_project copies it in; see src/genie_ontology_modeled_knowledge_sdk_vendored/README.md)"
        ) from exc


mv_yaml, manifest = load_sdk(args["bundle_root"])

In [ ]:
_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
_SQL_SOURCE = re.compile(r"^\s*(select|with)\b", re.IGNORECASE | re.DOTALL)


def quote(name: str) -> str:
    if _IDENT.match(name):
        return name
    if "`" in name:
        raise ValueError(f"identifier {name!r} must not contain backticks")
    return f"`{name}`"


def relation(source: str) -> str:
    """A metric view `source` is a table name or a SQL query."""
    source = source.strip()
    return f"({source})" if _SQL_SOURCE.match(source) else source


def flatten(joins, out=None):
    """Joins in dependency order: each parent before its nested joins."""
    out = [] if out is None else out
    for j in joins or []:
        out.append(j)
        flatten(j.get("joins"), out)
    return out


def join_clause(j: dict) -> str:
    if j.get("using"):
        cols = j["using"] if isinstance(j["using"], list) else [j["using"]]
        cond = "USING (" + ", ".join(cols) + ")"
    else:
        cond = f"ON {j['on']}"
    return f"LEFT JOIN {relation(j['source'])} AS {quote(j['name'])} {cond}"


def count(sql: str) -> int:
    return spark.sql(sql).first()[0]

In [ ]:
root = Path(args["bundle_root"])
params = {k: quote(args[k]) for k in ("source_catalog", "source_schema", "target_catalog", "target_schema")}
entries = manifest.build_manifest(root / args["metric_views_dir"], bundle_root=root)

results = []
for e in entries:
    doc = mv_yaml.load_mv_text(mv_yaml.substitute((root / e["yaml_path"]).read_text(), params))
    joins = flatten(doc.get("joins"))
    if not joins:
        results.append({"name": e["name"], "status": "no_joins"})
        continue
    names = [j.get("name") for j in joins]
    dupes = sorted({n for n in names if names.count(n) > 1})
    if dupes:
        results.append({"name": e["name"], "status": "error", "error": f"join names reused across nesting levels: {dupes}"})
        continue

    base_from = f"FROM {relation(doc['source'])} AS source"
    result = {"name": e["name"], "status": "pass", "joins": len(joins)}
    try:
        result["fact_rows"] = base = count(f"SELECT COUNT(*) {base_from}")
        clauses = []
        for j in joins:
            clauses.append(join_clause(j))
            n = count(f"SELECT COUNT(*) {base_from} " + " ".join(clauses))
            if n > base:
                result.update(status="inflated", join=j["name"], joined_rows=n)
                break
    except Exception as exc:  # recorded and re-raised below as one failure
        result.update(status="error", error=str(exc).splitlines()[0][:500])
    results.append(result)
    print(json.dumps(result))

In [ ]:
failed = [r for r in results if r["status"] in ("inflated", "error")]
summary = {
    "metric_views": len(results),
    "checked": sum(1 for r in results if r["status"] != "no_joins"),
    "failed": [{k: r.get(k) for k in ("name", "status", "join", "fact_rows", "joined_rows", "error")} for r in failed],
}
dbutils.jobs.taskValues.set(key="summary", value=summary)
print(json.dumps(summary, indent=2))
if failed:
    raise AssertionError(f"Join cardinality check failed for {len(failed)} metric view(s): " + json.dumps(summary["failed"])[:4000])
dbutils.notebook.exit(json.dumps(summary))